# MedSAM2 — ARCADE Dataset Evaluation

**Dataset:** ARCADE MICCAI 2023 — 1,200 X-ray coronary angiogram images, 512×512px  
**Source:** [Zenodo 10390295](https://zenodo.org/records/10390295)  
**Pipeline:** Same as DCA1 experiments — click prompt, frozen encoder, 20 epochs  
**Goal:** Establish whether 10× more data (1,000 vs 107 training images) pushes Dice past 0.75+

**Bucket:** `gs://coronary-angio-v2`

## Cell 1 — Environment Setup

In [ ]:
import os, subprocess, sys

subprocess.run(['sudo','apt-get','update','-q'], check=True)
subprocess.run(['sudo','apt-get','install','-y','ffmpeg','p7zip-full','unzip'], check=True)
subprocess.run([sys.executable,'-m','pip','install',
    'torch==2.5.1','torchvision==0.20.1',
    '--index-url','https://download.pytorch.org/whl/cu121','-q'], check=True)
subprocess.run([sys.executable,'-m','pip','install',
    'numpy>=2.0.1','opencv-python','pillow','matplotlib','tqdm','scipy','pycocotools','-q'], check=True)

if not os.path.exists('/home/jupyter/MedSAM2'):
    subprocess.run(['git','clone','https://github.com/bowang-lab/MedSAM2.git',
                    '/home/jupyter/MedSAM2'], check=True)
try:
    import sam2
except ImportError:
    subprocess.run([sys.executable,'-m','pip','install','-e','.', '-q'],
                   cwd='/home/jupyter/MedSAM2', check=True)

sys.path.insert(0, '/home/jupyter/MedSAM2')
print('Environment ready')

## Cell 2 — Checkpoint + GPU

In [ ]:
import torch

BUCKET     = 'gs://coronary-angio-v2'
CKPT_DIR   = '/home/jupyter/MedSAM2/checkpoints'
CHECKPOINT = os.path.join(CKPT_DIR, 'MedSAM2_latest.pt')
CONFIG     = 'configs/sam2.1_hiera_t512.yaml'
MODEL_SIZE = 512
HIRES      = MODEL_SIZE // 4
BB_FEAT_SIZES = [[HIRES//(2**k)]*2 for k in range(3)]

os.makedirs(CKPT_DIR, exist_ok=True)
if not os.path.exists(CHECKPOINT):
    subprocess.run(['bash','download.sh'], cwd='/home/jupyter/MedSAM2')
else:
    print('Checkpoint present')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device} | GPU: {torch.cuda.get_device_name(0)}')

## Cell 3 — Download ARCADE from Zenodo

ARCADE is freely available on Zenodo (no login required).  
**arcade.zip** is 451MB — download takes ~2 min on the instance.

**What's inside:**  
- `syntax/train/images/` — 1,000 PNG X-ray images (512×512)  
- `syntax/train/annotations/train.json` — COCO-format polygon annotations  
- `syntax/val/images/` — 200 PNG images  
- `syntax/val/annotations/val.json` — COCO-format polygon annotations  

**Annotation format:** COCO polygon segmentation with 26 categories (SYNTAX score coronary segments).  
We merge all 26 into a single binary vessel mask (foreground=vessel, background=everything else).

In [ ]:
GCS_ARCADE       = f'{BUCKET}/datasets/arcade'
PROC_TRAIN_IMGS  = '/tmp/arcade_proc/train/images'
PROC_TRAIN_MASKS = '/tmp/arcade_proc/train/masks'
PROC_VAL_IMGS    = '/tmp/arcade_proc/val/images'
PROC_VAL_MASKS   = '/tmp/arcade_proc/val/masks'

# Check if processed data already exists in GCS
check = subprocess.run(['gsutil', 'ls', f'{GCS_ARCADE}/train/images/'],
                       capture_output=True, text=True)
DATA_IN_GCS = check.returncode == 0 and bool(check.stdout.strip())

if DATA_IN_GCS:
    print('✓ Processed ARCADE found in GCS — syncing to instance (no Zenodo download)...')
    for split in ['train', 'val']:
        for folder in ['images', 'masks']:
            local_dir = f'/tmp/arcade_proc/{split}/{folder}'
            os.makedirs(local_dir, exist_ok=True)
            subprocess.run(['gsutil', '-m', 'rsync', '-r',
                            f'{GCS_ARCADE}/{split}/{folder}/', local_dir + '/'], check=True)
    print('Data ready from GCS.')
    NEED_CONVERT = False
else:
    print('No GCS cache — downloading ARCADE from Zenodo (~451 MB)...')
    ARCADE_DIR = '/tmp/arcade'
    ARCADE_ZIP = '/tmp/arcade.zip'
    if not os.path.exists(ARCADE_DIR) or not os.listdir(ARCADE_DIR):
        subprocess.run([
            'wget', '-q', '--show-progress',
            'https://zenodo.org/records/10390295/files/arcade.zip?download=1',
            '-O', ARCADE_ZIP
        ], check=True)
        print('Extracting...')
        os.makedirs(ARCADE_DIR, exist_ok=True)
        subprocess.run(['unzip', '-q', ARCADE_ZIP, '-d', ARCADE_DIR], check=True)
        os.remove(ARCADE_ZIP)
        print('Downloaded and extracted.')
    else:
        print('Arcade zip already extracted locally.')
    NEED_CONVERT = True

print(f'NEED_CONVERT={NEED_CONVERT}')

## Cell 4 — Convert COCO Annotations → Binary PNG Masks

ARCADE annotations are stored in COCO JSON format with polygon segmentations.  
Each image has up to 26 annotated vessel segments (LAD, LCX, RCA branches etc.).

**Conversion:** For each image, draw all polygon annotations onto a binary canvas.  
Any pixel covered by any vessel segment becomes foreground (255).  
This gives us one clean binary mask per image — same format as DCA1.

**Why merge all 26 categories?**  
Our model does binary segmentation (vessel vs background).  
The category breakdown (which specific branch) is not used here — future work.

In [ ]:
if NEED_CONVERT:
    SYNTAX_DIR = None
    for candidate in [
        os.path.join(ARCADE_DIR, 'arcade', 'syntax'),
        os.path.join(ARCADE_DIR, 'syntax'),
    ]:
        if os.path.exists(candidate):
            SYNTAX_DIR = candidate
            break
    if SYNTAX_DIR is None:
        for root, dirs, files in os.walk(ARCADE_DIR):
            if 'syntax' in dirs:
                SYNTAX_DIR = os.path.join(root, 'syntax')
                break
    print(f'SYNTAX_DIR: {SYNTAX_DIR}')
    print('Contents:', os.listdir(SYNTAX_DIR))

In [ ]:
import json, numpy as np
from PIL import Image, ImageDraw
import glob
from tqdm import tqdm

def coco_to_binary_masks(img_dir, ann_json, out_img_dir, out_mask_dir):
    """Convert COCO polygon annotations to binary PNG masks. All vessel segments merged."""
    os.makedirs(out_img_dir,  exist_ok=True)
    os.makedirs(out_mask_dir, exist_ok=True)
    with open(ann_json) as f:
        coco = json.load(f)
    id2img = {img['id']: img for img in coco['images']}
    from collections import defaultdict
    img_anns = defaultdict(list)
    for ann in coco['annotations']:
        img_anns[ann['image_id']].append(ann)
    converted = 0
    for img_id, img_info in tqdm(id2img.items()):
        fname   = img_info['file_name']
        H, W    = img_info['height'], img_info['width']
        src_img = os.path.join(img_dir, fname)
        if not os.path.exists(src_img):
            continue
        img_pil = Image.open(src_img).convert('RGB')
        stem    = os.path.splitext(fname)[0]
        img_pil.save(os.path.join(out_img_dir, f'{stem}.png'))
        mask = Image.new('L', (W, H), 0)
        draw = ImageDraw.Draw(mask)
        for ann in img_anns[img_id]:
            for seg in ann.get('segmentation', []):
                if len(seg) >= 6:
                    poly = [(seg[i], seg[i+1]) for i in range(0, len(seg)-1, 2)]
                    draw.polygon(poly, fill=255)
        mask.save(os.path.join(out_mask_dir, f'{stem}.png'))
        converted += 1
    return converted

if NEED_CONVERT:
    TRAIN_IMG_IN   = os.path.join(SYNTAX_DIR, 'train', 'images')
    TRAIN_ANN_JSON = None
    for candidate in [
        os.path.join(SYNTAX_DIR, 'train', 'annotations', 'train.json'),
        os.path.join(SYNTAX_DIR, 'train', 'annotations.json'),
        os.path.join(SYNTAX_DIR, 'annotations', 'train.json'),
    ]:
        if os.path.exists(candidate):
            TRAIN_ANN_JSON = candidate
            break
    if TRAIN_ANN_JSON is None:
        for root, dirs, files in os.walk(os.path.join(SYNTAX_DIR, 'train')):
            for f in files:
                if f.endswith('.json'):
                    TRAIN_ANN_JSON = os.path.join(root, f); break
    print(f'Train images: {TRAIN_IMG_IN}')
    print(f'Train annotations: {TRAIN_ANN_JSON}')
    n = coco_to_binary_masks(TRAIN_IMG_IN, TRAIN_ANN_JSON, PROC_TRAIN_IMGS, PROC_TRAIN_MASKS)
    print(f'Converted {n} training images')

In [ ]:
if NEED_CONVERT:
    VAL_IMG_IN   = os.path.join(SYNTAX_DIR, 'val', 'images')
    VAL_ANN_JSON = None
    for candidate in [
        os.path.join(SYNTAX_DIR, 'val', 'annotations', 'val.json'),
        os.path.join(SYNTAX_DIR, 'val', 'annotations.json'),
        os.path.join(SYNTAX_DIR, 'annotations', 'val.json'),
    ]:
        if os.path.exists(candidate):
            VAL_ANN_JSON = candidate
            break
    if VAL_ANN_JSON is None:
        for root, dirs, files in os.walk(os.path.join(SYNTAX_DIR, 'val')):
            for f in files:
                if f.endswith('.json'):
                    VAL_ANN_JSON = os.path.join(root, f); break
    n = coco_to_binary_masks(VAL_IMG_IN, VAL_ANN_JSON, PROC_VAL_IMGS, PROC_VAL_MASKS)
    print(f'Converted {n} val images')

    # Upload processed data to GCS so future runs skip Zenodo entirely
    print(f'\nUploading processed ARCADE to {GCS_ARCADE} ...')
    for split in ['train', 'val']:
        for folder in ['images', 'masks']:
            local_dir = f'/tmp/arcade_proc/{split}/{folder}'
            subprocess.run(['gsutil', '-m', 'rsync', '-r',
                            local_dir + '/', f'{GCS_ARCADE}/{split}/{folder}/'], check=True)
    print(f'✓ ARCADE images + masks saved to GCS — future runs will skip the download.')

## Cell 5 — Verify Pairs

In [ ]:
import matplotlib.pyplot as plt

train_img_paths  = sorted(glob.glob(f'{PROC_TRAIN_IMGS}/*.png'))
train_mask_paths = sorted(glob.glob(f'{PROC_TRAIN_MASKS}/*.png'))
test_img_paths   = sorted(glob.glob(f'{PROC_VAL_IMGS}/*.png'))
test_mask_paths  = sorted(glob.glob(f'{PROC_VAL_MASKS}/*.png'))

print(f'Train: {len(train_img_paths)} images | Test (val): {len(test_img_paths)} images')

# Check foreground fraction
fg_list = []
for mp in train_mask_paths[:50]:   # spot check 50
    m = np.array(Image.open(mp).convert('L').resize((256,256), Image.NEAREST))
    fg_list.append((m > 0).mean())
print(f'Foreground fraction: {np.mean(fg_list)*100:.1f}%')

# Visual spot check
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for col in range(4):
    idx = col * (len(train_img_paths)//4)
    img = np.array(Image.open(train_img_paths[idx]).convert('L'))
    msk = np.array(Image.open(train_mask_paths[idx]).convert('L'))
    axes[0,col].imshow(img, cmap='gray'); axes[0,col].set_title(os.path.basename(train_img_paths[idx])); axes[0,col].axis('off')
    axes[1,col].imshow(img, cmap='gray'); axes[1,col].imshow(msk>0, alpha=0.4, cmap='Reds'); axes[1,col].axis('off')
axes[0,0].set_ylabel('Image'); axes[1,0].set_ylabel('Image + Mask')
plt.suptitle('ARCADE train samples — verify vessel masks look correct', fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
def centroid_click(mask_np):
    ys, xs = np.where(mask_np > 0)
    if len(ys) == 0:
        return (mask_np.shape[1]//2, mask_np.shape[0]//2)
    return (int(xs.mean()), int(ys.mean()))

def dice_score(pred, gt, smooth=1e-5):
    p, g = pred.astype(float), gt.astype(float)
    return (2*(p*g).sum() + smooth) / (p.sum() + g.sum() + smooth)

def iou_score(pred, gt, smooth=1e-5):
    i = (pred.astype(bool) & gt.astype(bool)).sum()
    u = (pred.astype(bool) | gt.astype(bool)).sum()
    return (i + smooth) / (u + smooth)

print('Helpers ready')

## Cell 6 — Zero-Shot Baseline on Val Split

200 val images, centroid click. Expected: similar to DCA1 zero-shot (~0.10)  
since the modality is the same X-ray fluoroscopy that MedSAM2 has never been trained on.

In [ ]:
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

zs_model     = build_sam2(CONFIG, CHECKPOINT, device=device)
zs_predictor = SAM2ImagePredictor(zs_model)

zs_results = []
print(f'Zero-shot on {len(test_img_paths)} val images...')
for img_path, mask_path in tqdm(zip(test_img_paths, test_mask_paths), total=len(test_img_paths)):
    img_rgb = np.array(Image.open(img_path).convert('RGB'))
    gt_mask = np.array(Image.open(mask_path).convert('L')) > 0
    cx, cy  = centroid_click(gt_mask)
    with torch.inference_mode(), torch.autocast(device_type=device, dtype=torch.bfloat16):
        zs_predictor.set_image(img_rgb)
        preds, _, _ = zs_predictor.predict(
            point_coords=np.array([[cx, cy]]), point_labels=np.array([1]),
            multimask_output=False)
    pred_mask = preds[0]
    zs_results.append({'dice': dice_score(pred_mask, gt_mask),
                       'iou':  iou_score(pred_mask, gt_mask),
                       'pred': pred_mask, 'gt': gt_mask, 'img': img_rgb,
                       'path': img_path})

zs_dice = [r['dice'] for r in zs_results]
zs_iou  = [r['iou']  for r in zs_results]
print(f'Zero-Shot | Dice: {np.mean(zs_dice):.3f} ± {np.std(zs_dice):.3f} | IoU: {np.mean(zs_iou):.3f}')

## Cell 7 — Fine-Tune on 1,000 ARCADE Training Images

Same architecture as DCA1 experiments:  
- Frozen image encoder  
- Trainable mask decoder + prompt encoder  
- Click prompt (random foreground pixel)  
- Dice + weighted BCE loss  
- 20 epochs, LR 5e-5

In [ ]:
import torch.nn as nn, torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torch.cuda.amp import GradScaler

class ArcadeDataset(Dataset):
    def __init__(self, img_paths, mask_paths):
        self.img_paths  = img_paths
        self.mask_paths = mask_paths
        self.transform  = transforms.Compose([
            transforms.Resize((MODEL_SIZE, MODEL_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
        ])
    def __len__(self): return len(self.img_paths)
    def __getitem__(self, idx):
        img      = Image.open(self.img_paths[idx]).convert('RGB')
        mask_pil = Image.open(self.mask_paths[idx]).convert('L')
        mask_np  = np.array(mask_pil)
        img_t    = self.transform(img)
        mask_256 = np.array(mask_pil.resize((256,256), Image.NEAREST))
        mask_t   = torch.from_numpy((mask_256 > 0).astype(np.float32)).unsqueeze(0)
        ys, xs   = np.where(mask_np > 0)
        if len(ys) > 0:
            pick = np.random.randint(len(ys))
            py, px = int(ys[pick]), int(xs[pick])
        else:
            py, px = mask_np.shape[0]//2, mask_np.shape[1]//2
        h, w  = mask_np.shape
        point = torch.tensor([px/w*MODEL_SIZE, py/h*MODEL_SIZE], dtype=torch.float32)
        return img_t, mask_t, point

# pos_weight from training masks
fg_list = [
    (np.array(Image.open(mp).convert('L').resize((256,256), Image.NEAREST)) > 0).mean()
    for mp in train_mask_paths[:200]   # sample 200 for speed
]
fg_mean    = float(np.mean(fg_list))
pos_weight = torch.tensor([(1-fg_mean)/fg_mean], device=device)
print(f'Foreground: {fg_mean*100:.1f}%  |  pos_weight: {pos_weight.item():.1f}x')

train_ds     = ArcadeDataset(train_img_paths, train_mask_paths)
train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=2, pin_memory=True)
print(f'Train: {len(train_ds)} images | {len(train_loader)} batches/epoch')

In [ ]:
model = build_sam2(CONFIG, CHECKPOINT, device=device)
for name, param in model.named_parameters():
    param.requires_grad = ('image_encoder' not in name)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,}  ({100*trainable/total:.1f}%)')

In [ ]:
CKPT_PATH = '/tmp/medsam2_arcade.pt'
EPOCHS    = 20

def dice_loss(logits, target, smooth=1e-5):
    pred  = torch.sigmoid(logits)
    inter = (pred * target).sum(dim=(2,3))
    union = pred.sum(dim=(2,3)) + target.sum(dim=(2,3))
    return (1 - (2*inter + smooth)/(union + smooth)).mean()

def combined_loss(logits, target, pw):
    return dice_loss(logits, target) + nn.BCEWithLogitsLoss(pos_weight=pw)(logits, target)

optimizer = optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=5e-5, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler    = GradScaler()
best_loss = float('inf')
train_losses = []

print(f'Training {EPOCHS} epochs on {len(train_ds)} ARCADE images...')

for epoch in range(EPOCHS):
    model.train()
    ep_loss = 0.0
    for imgs, masks, points in tqdm(train_loader, desc=f'Ep {epoch+1}/{EPOCHS}', leave=False):
        optimizer.zero_grad()
        B = imgs.shape[0]
        imgs_dev, points_dev = imgs.to(device), points.to(device)

        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            with torch.no_grad():
                backbone_out = model.forward_image(imgs_dev)
                _, vision_feats, _, _ = model._prepare_backbone_features(backbone_out)
                if model.directly_add_no_mem_embed:
                    vision_feats[-1] = vision_feats[-1] + model.no_mem_embed
            feats = [feat.permute(1,2,0).view(B,-1,*fs)
                     for feat, fs in zip(vision_feats[::-1], BB_FEAT_SIZES[::-1])][::-1]
            image_embed, high_res_feats = feats[-1], feats[:-1]

            all_logits = []
            for i in range(B):
                pt       = points_dev[i].reshape(1,1,2)
                pt_label = torch.ones(1,1, dtype=torch.int, device=device)
                sparse_emb, dense_emb = model.sam_prompt_encoder(
                    points=(pt, pt_label), boxes=None, masks=None)
                low_res_masks, _, _, _ = model.sam_mask_decoder(
                    image_embeddings=image_embed[i].unsqueeze(0),
                    image_pe=model.sam_prompt_encoder.get_dense_pe(),
                    sparse_prompt_embeddings=sparse_emb,
                    dense_prompt_embeddings=dense_emb,
                    multimask_output=False, repeat_image=False,
                    high_res_features=[f[i].unsqueeze(0) for f in high_res_feats])
                all_logits.append(low_res_masks)

            logits  = torch.cat(all_logits, dim=0)
            targets = masks.to(device)
            if targets.shape[-2:] != logits.shape[-2:]:
                targets = F.interpolate(targets.float(), size=logits.shape[-2:], mode='nearest')
            loss = combined_loss(logits, targets, pos_weight)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
        scaler.step(optimizer)
        scaler.update()
        ep_loss += loss.item()

    ep_loss /= len(train_loader)
    train_losses.append(ep_loss)
    scheduler.step()
    print(f'Epoch {epoch+1:2d} | Loss: {ep_loss:.4f}')
    if ep_loss < best_loss:
        best_loss = ep_loss
        torch.save(model.state_dict(), CKPT_PATH)
        print(f'           ★ saved')

print(f'\nBest loss: {best_loss:.4f}')
!gsutil cp {CKPT_PATH} {BUCKET}/checkpoints/medsam2_arcade.pt

plt.figure(figsize=(8,4))
plt.plot(range(1,EPOCHS+1), train_losses, 'b-o', markersize=4)
plt.xlabel('Epoch'); plt.ylabel('Loss (Dice + wBCE)')
plt.title(f'ARCADE Fine-Tuning — {len(train_ds)} images, {EPOCHS} epochs')
plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig('/tmp/arcade_loss.png', dpi=100); plt.show()
!gsutil cp /tmp/arcade_loss.png {BUCKET}/results/arcade_loss.png

## Cell 8 — Fine-Tuned Inference on 200 Val Images

In [ ]:
ft_model = build_sam2(CONFIG, CHECKPOINT, device=device)
incomp   = ft_model.load_state_dict(torch.load(CKPT_PATH, map_location=device), strict=False)
print(f'Missing: {len(incomp.missing_keys)} | Unexpected: {len(incomp.unexpected_keys)}')
assert len(incomp.missing_keys) == 0, '⚠ Checkpoint load failed!'
print('✓ Clean checkpoint load')
ft_model.eval()
ft_predictor = SAM2ImagePredictor(ft_model)

ft_results = []
print(f'Fine-tuned inference on {len(test_img_paths)} val images...')
for img_path, mask_path in tqdm(zip(test_img_paths, test_mask_paths), total=len(test_img_paths)):
    img_rgb = np.array(Image.open(img_path).convert('RGB'))
    gt_mask = np.array(Image.open(mask_path).convert('L')) > 0
    cx, cy  = centroid_click(gt_mask)
    with torch.inference_mode(), torch.autocast(device_type=device, dtype=torch.bfloat16):
        ft_predictor.set_image(img_rgb)
        preds, _, _ = ft_predictor.predict(
            point_coords=np.array([[cx, cy]]), point_labels=np.array([1]),
            multimask_output=False)
    pred_mask = preds[0]
    ft_results.append({'dice': dice_score(pred_mask, gt_mask),
                       'iou':  iou_score(pred_mask, gt_mask),
                       'pred': pred_mask, 'gt': gt_mask, 'img': img_rgb})

ft_dice = [r['dice'] for r in ft_results]
ft_iou  = [r['iou']  for r in ft_results]
print(f'\n=== ARCADE Results (200 val images) ===')
print(f'Zero-Shot              | Dice: {np.mean(zs_dice):.3f} ± {np.std(zs_dice):.3f} | IoU: {np.mean(zs_iou):.3f}')
print(f'Fine-Tuned (1000 imgs) | Dice: {np.mean(ft_dice):.3f} ± {np.std(ft_dice):.3f} | IoU: {np.mean(ft_iou):.3f}')
print(f'\nDCA1 reference (107 train / 27 test): ZS=0.096  FT=0.702')

In [ ]:
# Visual panel — 6 samples (worst → best fine-tuned Dice)
ft_sorted = sorted(ft_results, key=lambda r: r['dice'])
zs_sorted = sorted(zs_results, key=lambda r: r['dice'])
show_ft   = [ft_sorted[0], ft_sorted[len(ft_sorted)//4], ft_sorted[len(ft_sorted)//2],
             ft_sorted[3*len(ft_sorted)//4], ft_sorted[-2], ft_sorted[-1]]
show_zs   = [zs_results[ft_results.index(r)] for r in show_ft]

fig, axes = plt.subplots(3, 6, figsize=(26, 11))
for col, (fr, zr) in enumerate(zip(show_ft, show_zs)):
    img = fr['img'][:,:,0]
    axes[0,col].imshow(img, cmap='gray')
    axes[0,col].imshow(fr['gt'], alpha=0.35, cmap='Greens')
    axes[0,col].set_title(f'D={fr["dice"]:.2f}'); axes[0,col].axis('off')
    axes[1,col].imshow(img, cmap='gray')
    axes[1,col].imshow(zr['pred'], alpha=0.45, cmap='Reds')
    axes[1,col].set_title(f'ZS D={zr["dice"]:.2f}'); axes[1,col].axis('off')
    axes[2,col].imshow(img, cmap='gray')
    axes[2,col].imshow(fr['pred'], alpha=0.45, cmap='Blues')
    axes[2,col].set_title(f'FT D={fr["dice"]:.2f}'); axes[2,col].axis('off')
for row, label in enumerate(['GT overlay','Zero-Shot','Fine-Tuned (ARCADE)']):
    axes[row,0].set_ylabel(label, fontsize=10)
plt.suptitle(f'ARCADE | ZS={np.mean(zs_dice):.3f}  FT={np.mean(ft_dice):.3f} | 1000 train, 200 test', fontsize=12)
plt.tight_layout()
plt.savefig('/tmp/arcade_comparison.png', dpi=120, bbox_inches='tight')
plt.show()
!gsutil cp /tmp/arcade_comparison.png {BUCKET}/results/arcade_comparison.png
print('Results saved to GCS')

## Results — Update results_log.md

Update row #6 in `results_log.md` with the numbers from this run.  
Key comparison:
- DCA1 107 train → Dice 0.702
- ARCADE 1000 train → Dice **[your result here]**

If ARCADE gives >0.75, the bottleneck was data volume.  
If ARCADE gives similar (~0.70), the bottleneck is the frozen encoder not seeing X-ray.